# Cost-Benefit Score (Phase 1 baseline)

Objective: compute a cost-benefit score combining player performance
(2024 season) and market value, as a traditional efficiency-based scouting
baseline — the type of system this project's underlying thesis critiques,
built here as a technical foundation (see README).

## Known limitation: temporal mismatch between value and performance

Market value data (Transfermarkt) reflects the most recent valuation
available, not the player's value specifically in 2024 — while performance
stats (Eduardo Palmieri) are specifically from the 2024 season. This means
the cost-benefit score answers "is this player's current valuation
consistent with their 2024 performance?" rather than "was this player
underpriced relative to their 2024 performance at the time."

In [1]:
import pandas as pd

## Loading cleaned data

In [2]:
df = pd.read_csv('../data/processed/players_final_2024.csv')

## Filtering to players with matched stats

In [3]:
df_scored = df[df['stats_player_name'].notna()]
df_scored.shape

(326, 39)

## Grouping players by position

In [4]:
df_scored['stats_position'].unique()

<StringArray>
[   'GK',    'LM',    'CB',    'AM',    'RB',    'CM',    'RM', 'FW,AM',
 'AM,DM',    'RW',    'FW',    'LB',    'WB',    'LW', 'RM,FW', 'FW,LW',
    'DM', 'CB,CM', 'RM,LM', 'FW,LM', 'WB,CM', 'LM,RM', 'DM,CB', 'RW,LW',
 'CM,RM', 'FW,RW', 'LB,CM', 'CB,RW', 'LM,DM', 'LM,FW', 'RB,CB', 'RM,LW']
Length: 32, dtype: str

In [5]:
df_scored['sub_position'].unique()

<StringArray>
[        'Goalkeeper', 'Attacking Midfield',        'Centre-Back',
         'Right-Back',        'Left Winger',   'Central Midfield',
     'Centre-Forward',       'Right Winger',          'Left-Back',
 'Defensive Midfield',                  nan]
Length: 11, dtype: str

In [6]:
position_groups = {
    'Goalkeeper': ['Goalkeeper'],
    'Centre-Back': ['Centre-Back'],
    'Full-Back': ['Right-Back', 'Left-Back'],
    'Defensive Midfielder': ['Defensive Midfield'],
    'Central Midfielder': ['Central Midfield'],
    'Attacking Midfielder': ['Attacking Midfield'],
    'Winger': ['Right Winger', 'Left Winger'],
    'Centre-Forward': ['Centre-Forward'],
}

def map_position_group(position):
    for group, positions in position_groups.items():
        if position in positions:
            return group
    return 'Unknown'

df_scored['position_group'] = df_scored['sub_position'].apply(map_position_group)
df_scored.loc[df_scored['player_name'] == 'Pedro Rangel', 'position_group'] = 'Goalkeeper'
df_scored['position_group'].value_counts()

position_group
Centre-Back             62
Full-Back               50
Winger                  47
Defensive Midfielder    41
Centre-Forward          38
Attacking Midfielder    30
Goalkeeper              29
Central Midfielder      29
Name: count, dtype: int64

## Normalizing metrics per 90 minutes

In [7]:
df_scored_filtered = df_scored[df_scored['minutes_played'] >= 900]
df_scored_filtered.shape

(198, 40)

In [8]:
metrics_to_normalize = ['goals', 'assists', 'xG', 'xAG', 'progressive_passes',
                         'progressive_carries', 'dribbles_successful', 'shot_creating_actions',
                         'goal_creating_actions', 'tackles', 'blocks']

for metric in metrics_to_normalize:
    df_scored_filtered[f'{metric}_per90'] = (df_scored_filtered[metric] / df_scored_filtered['minutes_played']) * 90

df_scored_filtered[[f'{m}_per90' for m in metrics_to_normalize]].head()

,goals_per90,assists_per90,xG_per90,xAG_per90,progressive_passes_per90,progressive_carries_per90,dribbles_successful_per90,shot_creating_actions_per90,goal_creating_actions_per90,tackles_per90,blocks_per90
0,0.000000,0.000000,0.007673,0.005115,0.332481,0.281330,0.076726,0.358056,0.025575,0.204604,0.102302
1,0.179283,0.239044,0.095618,0.256972,4.362550,1.374502,0.717131,4.482072,0.239044,0.597610,0.478088
2,0.000000,0.000000,0.035714,0.050000,2.428571,0.142857,0.000000,0.928571,0.000000,0.928571,0.714286
3,0.221311,0.221311,0.125410,0.184426,7.450820,1.770492,1.254098,4.500000,0.663934,0.885246,0.147541
4,0.075885,0.000000,0.022766,0.045531,2.428331,0.151771,0.075885,0.682968,0.151771,1.290051,0.531197


## Excluding goalkeepers (no suitable metrics available) and Luiz Gustavo (Database error)

### Note: goalkeepers excluded from scoring

None of the available performance metrics (goals, assists, progressive
actions, outfield defensive actions) meaningfully evaluate goalkeeper
performance. Proper goalkeeper analysis would require metrics like saves,
save percentage, or goals prevented — not present in this dataset.
Goalkeepers are excluded from the cost-benefit score in this version;
adding them is a documented next step, pending a suitable data source.

In [9]:
df_outfield = df_scored_filtered[df_scored_filtered['position_group'] != 'Goalkeeper']
df_outfield['position_group'].value_counts()

position_group
Centre-Back             41
Full-Back               35
Winger                  26
Attacking Midfielder    21
Centre-Forward          21
Defensive Midfielder    21
Central Midfielder      17
Name: count, dtype: int64

In [10]:
df_outfield = df_outfield[df_outfield['player_id'] != 1220468]
df_outfield.shape

(181, 51)

## Defining position-specific weights

In [11]:
position_weights = {
    'Winger': {
        'xAG_per90': 2.5,
        'assists_per90': 1.5,
        'dribbles_successful_per90': 2.0,
        'shot_creating_actions_per90': 2.0,
        'goals_per90': 1.5,
        'xG_per90': 2.0,
        'goal_creating_actions_per90': 1.5,
    },
    'Centre-Forward': {
        'xG_per90': 3.0,
        'goals_per90': 2.5,
        'goal_creating_actions_per90': 1.5,
        'assists_per90': 1.0,
        'xAG_per90': 1.0,
        'shot_creating_actions_per90': 1.0,

    },
    'Attacking Midfielder': {
        'xAG_per90': 3.0,
        'assists_per90': 2.0,
        'shot_creating_actions_per90': 2.5,
        'goal_creating_actions_per90': 2.0,
        'goals_per90': 1.5,
        'xG_per90': 1.5,
        'dribbles_successful_per90': 1.0,
    },
    'Central Midfielder': {
        'progressive_passes_per90': 3.0,
        'progressive_carries_per90': 1.5,
        'xAG_per90': 2.0,
        'assists_per90': 1.0,
        'shot_creating_actions_per90': 1.5,
        'tackles_per90': 1.5,
        'blocks_per90': 1.0,
    },
    'Defensive Midfielder': {
        'tackles_per90': 2.0,
        'blocks_per90': 1.5,
        'progressive_passes_per90': 3.0,
        'progressive_carries_per90': 1.0,
        'xAG_per90': 0.75,
        'assists_per90': 0.25,
    },
    'Full-Back': {
        'progressive_carries_per90': 2.5,
        'progressive_passes_per90': 2.5,
        'assists_per90': 1.5,
        'xAG_per90': 2.0,
        'tackles_per90': 1.5,
        'blocks_per90': 1.5,
        'dribbles_successful_per90': 1.0,
    },
    'Centre-Back': {
        'tackles_per90': 2.0,
        'blocks_per90': 2.0,
        'progressive_passes_per90': 2.5,
        'progressive_carries_per90': 1.0,
    },
}

## Calculating within-position percentiles

Comparing average performance_score across position groups confirms the
percentile-based approach resolved the scale mismatch seen with raw
weighted scores (where Central Midfielder averaged ~26 and Forward
averaged ~6, not directly comparable). After normalization, all position
groups average close to 0.52, with comparable spread — scores are now
meaningfully comparable across different positions

In [12]:
def calculate_percentiles(df, metrics, group_column='position_group'):

    df = df.copy()
    for metric in metrics:
        percentile_col = f'{metric}_percentile'
        df[percentile_col] = df.groupby(group_column)[metric].rank(pct=True)
    return df

In [13]:
df_outfield = calculate_percentiles(df_outfield, metrics_to_normalize)
df_outfield[[f'{m}_percentile' for m in metrics_to_normalize]].head()

,goals_percentile,assists_percentile,xG_percentile,xAG_percentile,progressive_passes_percentile,progressive_carries_percentile,dribbles_successful_percentile,shot_creating_actions_percentile,goal_creating_actions_percentile,tackles_percentile,blocks_percentile
1,0.428571,0.547619,0.238095,0.642857,0.333333,0.333333,0.261905,0.476190,0.214286,0.119048,0.166667
2,0.262500,0.350000,0.350000,0.750000,0.175000,0.162500,0.062500,0.437500,0.162500,0.100000,0.112500
3,0.428571,0.404762,0.285714,0.404762,0.571429,0.380952,0.476190,0.380952,0.523810,0.190476,0.047619
4,0.675000,0.350000,0.237500,0.687500,0.150000,0.162500,0.187500,0.225000,0.687500,0.175000,0.062500
5,0.925000,0.350000,0.875000,0.587500,0.825000,0.950000,0.562500,0.862500,0.425000,0.050000,0.350000


## Calculating weighted performance score

In [14]:
def calculate_position_score(row, weights_dict):
    
    weights = weights_dict.get(row['position_group'])
    if weights is None:
        return None
    score = 0
    total_weight = 0
    for metric, weight in weights.items():
        percentile_col = f'{metric}_percentile'
        score += row[percentile_col] * weight
        total_weight += weight
    return score / total_weight  # normalize by total weight used

In [15]:
per90_metrics = [f'{metric}_per90' for metric in metrics_to_normalize]

df_outfield = calculate_percentiles(df_outfield, per90_metrics)
df_outfield[[f'{m}_percentile' for m in per90_metrics]].head()

,goals_per90_percentile,assists_per90_percentile,xG_per90_percentile,xAG_per90_percentile,progressive_passes_per90_percentile,progressive_carries_per90_percentile,dribbles_successful_per90_percentile,shot_creating_actions_per90_percentile,goal_creating_actions_per90_percentile,tackles_per90_percentile,blocks_per90_percentile
1,0.47619,0.666667,0.238095,0.857143,0.380952,0.285714,0.333333,0.523810,0.333333,0.095238,0.190476
2,0.26250,0.350000,0.425000,0.800000,0.350000,0.125000,0.062500,0.650000,0.162500,0.100000,0.150000
3,0.52381,0.619048,0.380952,0.571429,0.904762,0.523810,0.619048,0.571429,0.857143,0.238095,0.047619
4,0.77500,0.350000,0.250000,0.775000,0.325000,0.175000,0.225000,0.375000,0.875000,0.375000,0.050000
5,0.95000,0.350000,0.875000,0.625000,0.950000,0.950000,0.500000,0.925000,0.400000,0.025000,0.325000


In [16]:
df_outfield['performance_score'] = df_outfield.apply(lambda row: calculate_position_score(row, position_weights), axis=1)

## Verifying weighted performance scores

In [17]:
df_outfield[['player_name', 'position_group', 'performance_score']].head(10)

,player_name,position_group,performance_score
1,Nenê,Attacking Midfielder,0.539683
2,Thiago Silva,Centre-Back,0.200000
3,Dimitri Payet,Attacking Midfielder,0.597884
4,Gabriel Mercado,Centre-Back,0.245000
5,David Luiz,Centre-Back,0.536667
6,Fagner,Full-Back,0.482286
9,Willian,Winger,0.214497
10,Everton Ribeiro,Attacking Midfielder,0.380952
14,Germán Cano,Centre-Forward,0.269048
15,Titi,Centre-Back,0.418333


In [18]:
df_outfield['performance_score'].describe()

count    181.000000
mean       0.519337
std        0.176685
min        0.089497
25%        0.406162
50%        0.518857
75%        0.628571
max        0.955908
Name: performance_score, dtype: float64

In [19]:
df_outfield[(df_outfield['position_group'] == 'Centre-Back') & ((df_outfield['player_name'] == 'Thiago Silva') | (df_outfield['player_name'] == 'David Luiz')) ].nlargest(10, 'performance_score')[['player_name', 'performance_score', 'minutes_played', 'tackles', 'blocks', 'progressive_passes', 'progressive_carries']]

,player_name,performance_score,minutes_played,tackles,blocks,progressive_passes,progressive_carries
5,David Luiz,0.536667,1756.0,11.0,19.0,104.0,32.0
2,Thiago Silva,0.200000,1260.0,13.0,10.0,34.0,2.0


In [20]:
df_outfield.groupby('position_group')['performance_score'].agg(['mean', 'std', 'min', 'max'])

,mean,std,min,max
position_group,,,,
Attacking Midfielder,0.523810,0.209535,0.149912,0.955908
Central Midfielder,0.529412,0.172663,0.263427,0.872123
Centre-Back,0.512500,0.176979,0.141667,0.916667
Centre-Forward,0.523810,0.218144,0.145238,0.952381
Defensive Midfielder,0.523810,0.151534,0.163866,0.866947
Full-Back,0.514286,0.142379,0.185143,0.801143
Winger,0.519231,0.193554,0.089497,0.806213


## Moneyball Score: combining performance, age, and market value

In [21]:
df_outfield['date_of_birth'] = pd.to_datetime(df_outfield['date_of_birth'])
df_outfield['age'] = (pd.Timestamp('2024-12-31') - df_outfield['date_of_birth']).dt.days // 365
df_outfield[['player_name', 'date_of_birth', 'age']].head()

,player_name,date_of_birth,age
1,Nenê,1981-07-19,43
2,Thiago Silva,1984-09-22,40
3,Dimitri Payet,1987-03-29,37
4,Gabriel Mercado,1987-03-18,37
5,David Luiz,1987-04-22,37


In [22]:
df_outfield = calculate_percentiles(df_outfield, ['market_value_in_eur', 'age'])
df_outfield[['player_name', 'position_group', 'market_value_in_eur_percentile', 'age_percentile']].head()

,player_name,position_group,market_value_in_eur_percentile,age_percentile
1,Nenê,Attacking Midfielder,0.058824,1.000000
2,Thiago Silva,Centre-Back,0.227273,1.000000
3,Dimitri Payet,Attacking Midfielder,0.529412,0.952381
4,Gabriel Mercado,Centre-Back,0.030303,0.962500
5,David Luiz,Centre-Back,0.227273,0.962500


In [23]:
def calculate_moneyball_score(row, performance_weight=0.6, value_weight=0.25, age_weight=0.15):
    """
    Combines performance, market value, and age into a single score.
    Performance: higher percentile is better.
    Market value and age: lower percentile is better (cheaper, younger),
    so we invert them (1 - percentile) before combining.
    """
    if pd.isna(row['performance_score']):
        return None
    
    value_score = 1 - row['market_value_in_eur_percentile']
    age_score = 1 - row['age_percentile']
    
    moneyball = (
        row['performance_score'] * performance_weight +
        value_score * value_weight +
        age_score * age_weight
    )
    return moneyball

df_outfield['moneyball_score'] = df_outfield.apply(calculate_moneyball_score, axis=1)
df_outfield[['player_name', 'position_group', 'performance_score', 'market_value_in_eur_percentile', 'age_percentile', 'moneyball_score']].sort_values('moneyball_score', ascending=False).head(15)

,player_name,position_group,performance_score,market_value_in_eur_percentile,age_percentile,moneyball_score
39,Tiquinho Soares,Centre-Forward,0.800000,0.194444,0.738095,0.720675
329,Ademir,Winger,0.798817,0.285714,0.634615,0.712669
260,Pedro,Centre-Forward,0.952381,0.833333,0.357143,0.709524
346,Bruno Fuchs,Centre-Back,0.885000,0.772727,0.200000,0.707818
279,Nahuel Ferraresi,Centre-Back,0.916667,0.833333,0.262500,0.702292
765,Gustavinho,Defensive Midfielder,0.550420,0.058824,0.095238,0.701261
466,Claudinho,Full-Back,0.729143,0.370968,0.300000,0.699744
32,Hulk,Centre-Forward,0.883333,0.333333,1.000000,0.696667
503,Moisés,Winger,0.806213,0.500000,0.500000,0.683728
379,Gabriel Menino,Central Midfielder,0.780051,0.566667,0.352941,0.673423


## Sanity check: club consistency for matched players

In [24]:
df_outfield[['player_name', 'current_club_name', 'stats_club']].sample(20)

,player_name,current_club_name,stats_club
84,André Ramalho,Sport Club Corinthians Paulista,Corinthians
40,Alan Patrick,Sport Club Internacional,Internacional
47,Manoel,Fluminense Football Club,Fluminense
82,Bernard,Clube Atlético Mineiro,Atlético Mineiro
23,Marinho,Esporte Clube Vitória,Fortaleza
583,Robert Renan,Club de Regatas Vasco da Gama,Internacional
255,José Luis Rodríguez,Club de Regatas Vasco da Gama,Vasco da Gama
3,Dimitri Payet,Club de Regatas Vasco da Gama,Vasco da Gama
258,Luan Cândido,Esporte Clube Vitória,Red Bull Bragantino
233,Thaciano,Santos Futebol Clube,Bahia


## Exporting final scored dataset

In [25]:
df_outfield.to_csv('../data/processed/players_scored_2024.csv', index=False)

## Conclusion

- Loaded the cleaned dataset (326 players with matched 2024 stats)
- Filtered to players with 900+ minutes played (198 players), to avoid
  small-sample noise in per-90 rates
- Normalized key performance metrics to per-90-minutes rates
- Classified players into 7 granular position groups (Goalkeeper,
  Centre-Back, Full-Back, Defensive/Central/Attacking Midfielder, Winger,
  Centre-Forward), reflecting distinct tactical roles
- Excluded goalkeepers (29 players) — no suitable performance metrics
  available in this dataset; documented as a next step
- Computed within-position percentile ranks for all performance metrics,
  resolving a scale mismatch found when using raw weighted scores (some
  position groups averaged ~6, others ~26 — not directly comparable).
  After percentile normalization, all position groups average ~0.52,
  confirming scores are now comparable across positions
- Combined performance (60%), market value (25%, inverted — cheaper is
  better), and age (15%, inverted — younger is better) into a single
  Moneyball Score per player
- Found and removed one homonym error undetected in notebook 04 (a
  different, much younger "Luiz Gustavo" incorrectly matched to the
  well-known veteran's data) — a reminder that name-based matching may
  still contain a small number of undetected errors
- Known limitation: market value reflects the most recent valuation, not
  specifically the player's 2024 value — the score answers "is current
  valuation consistent with 2024 performance?" rather than identifying
  historical underpricing
- Result exported to data/processed/players_scored_2024.csv, ready for
  clustering (player profile grouping) and the Power BI dashboard